In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt


In [2]:
data = pd.read_csv("train.csv")
data['date'] = pd.to_datetime(data['date'])

data = data.sort_values(['store','item','date'])


In [5]:
# Lag features (group-wise)
data['lag_1'] = data.groupby(['store','item'])['sales'].shift(1)
data['lag_7'] = data.groupby(['store','item'])['sales'].shift(7)
data['lag_30'] = data.groupby(['store','item'])['sales'].shift(30)

# Rolling features (correct method)
data['rolling_7'] = (
    data.groupby(['store','item'])['sales']
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

data['rolling_30'] = (
    data.groupby(['store','item'])['sales']
    .transform(lambda x: x.shift(1).rolling(30).mean())
)

# Calendar features
data['month'] = data['date'].dt.month
data['day_of_week'] = data['date'].dt.dayofweek

# Drop NA rows
data = data.dropna()


In [6]:
split_date = data['date'].quantile(0.8)

train = data[data['date'] <= split_date]
valid = data[data['date'] > split_date]


In [7]:
features = [
    'store','item',
    'lag_1','lag_7','lag_30',
    'rolling_7','rolling_30',
    'month','day_of_week'
]

X_train = train[features]
y_train = train['sales']

X_valid = valid[features]
y_valid = valid['sales']


In [8]:
model = xgb.XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

predictions = model.predict(X_valid)


In [9]:
mae = mean_absolute_error(y_valid, predictions)
rmse = np.sqrt(mean_squared_error(y_valid, predictions))

print("Panel Model MAE:", mae)
print("Panel Model RMSE:", rmse)


Panel Model MAE: 6.2568864822387695
Panel Model RMSE: 8.154104367113849
